# 5G Network Intelligence — Fixed MLP Blend Candidate Audit

## Development evidence and question

The blend chosen on inner1 reduced inner2 MAE 0.518% / 0.602% / 0.434% at 1/6/24h. It is fixed at 20% B for every horizon. Test whether the effect persists on reused validation; do not change these coefficients.

## Frozen audit procedure

Refit B through target hour 503 for the rounded median of development best_epoch+1 (B=10). No early stopping. Reuse the previously verified hurdle validation forecasts unchanged. Only after refitting, audit 504–671 on all original eligible beams. No hyperparameter search, calibration, final holdout replay or production promotion.


In [ ]:
RUN_TRAINING = False
SOURCE_COMMIT = '399c36e0d40a63b9fb59fdd63324bb6399b74968'


## Fixed protocol and pinned evidence

The saved selection and hurdle replay must match their pinned SHA256 values. The original hurdle is not retrained.


In [ ]:
AUDIT_PROTOCOL = {'protocol': 'fixed-hurdle-mlp-validation-audit-v1',
 'fit_end': 503,
 'audit_end': 671,
 'horizons': [1, 6, 24],
 'epoch_policy': 'rounded_median_of_two_development_best_epoch_plus_one',
 'expected_selected_variants': ['B'],
 'expected_fixed_epochs': {'B': 10},
 'selection_sha256': '283945dd2464f85e33bf095a70f1a34981158622ef573c76d95367a69d8f721e',
 'mlp_manifest_sha256': 'e9dd565ee26362c3ec2f79ff5061722df2a72c83ca3d434944c0f9042b115ba1',
 'hurdle_replay_sha256': 'ec7bc06e9c6d205276436328f64117ce3b7c0fc994f08dcca907a44893e41dc2',
 'early_stopping': 'forbidden',
 'coefficient_changes': 'forbidden',
 'hurdle_retraining': 'forbidden',
 'original_final_holdout': 'forbidden',
 'production_change': 'forbidden',
 'claim_scope': 'reused_validation_audit_not_independent_generalization'}


## Completed hosted source inputs

Attach Residual MLP, development blend and the original zero-inflated candidate audit outputs. A small MIT source attachment supplies the new fixed-epoch trainer and forecast arithmetic. No repository source dump is embedded.


In [ ]:
from pathlib import Path
import os,sys,shutil,subprocess,tarfile
if RUN_TRAINING:
    if not Path('/kaggle/working').is_dir():
        raise RuntimeError('Hosted Kaggle required')
    mlp_roots=[p.parent.parent for p in Path('/kaggle/input').rglob('temporal_mlp_development.csv') if (p.parent.parent/'artifacts/temporal_mlp/manifest.json').exists()]
    blend_roots=[p.parent.parent for p in Path('/kaggle/input').rglob('mlp_blend_confirmation.csv') if (p.parent.parent/'artifacts/mlp_blend/manifest.json').exists()]
    hurdle_replays=list(Path('/kaggle/input').rglob('hurdle_candidate_val.npz'))
    if len(mlp_roots)!=1 or len(blend_roots)!=1 or len(hurdle_replays)!=1:
        raise RuntimeError('Attach exactly one completed output for each required source')
    MLP=mlp_roots[0]/'artifacts/temporal_mlp'
    BLEND=blend_roots[0]/'artifacts/mlp_blend'
    HURDLE_REPLAY=hurdle_replays[0]
    PROJECT=Path('/kaggle/working/mlp_blend_candidate')
    PROJECT.mkdir(exist_ok=True)
    for folder in ('src','scripts','configs','data/processed'):
        shutil.copytree(mlp_roots[0]/folder,PROJECT/folder,dirs_exist_ok=True,ignore=shutil.ignore_patterns('__pycache__'))
    os.chdir(PROJECT)
    sys.path.insert(0,str(PROJECT))
    os.environ['SOURCE_GIT_COMMIT']=SOURCE_COMMIT
    Path('reports').mkdir(exist_ok=True)
    patches=list(Path('/kaggle/input').rglob('finalize_mlp_blend_candidate.py'))
    if len(patches)!=1:
        raise RuntimeError('Attach exactly one fixed-audit source')
    patch=patches[0].parent
    for name,folder in (('finalize_blend_candidate.py','scripts'),('finalize_mlp_blend_candidate.py','scripts'),('temporal_mlp.py','src/models'),('forecast_blend.py','src/models'),('mlp_blend_candidate_audit.yaml','configs')):
        shutil.copyfile(patch/name,Path(folder)/name)


## GPU and fitting controls

Use the hosted image's PyTorch. Missing CUDA stops fitting. Save every epoch's final state and training loss; do not choose a best epoch from validation. The last fixed epoch is the candidate.


In [ ]:
if RUN_TRAINING:
    import torch
    print('PyTorch',torch.__version__)
    if not torch.cuda.is_available():
        raise RuntimeError('Hosted CUDA GPU required')
    subprocess.run(['nvidia-smi'],check=True)


## Fixed refit and audit

One MLP-B fit, ten fixed epochs in total. After fitting, construct the declared mixture and retain all component/audit scores. A validation gain is exploratory evidence, not an unseen-test guarantee.


In [ ]:
if RUN_TRAINING:
    command=[sys.executable,'-u','-m','scripts.finalize_mlp_blend_candidate','--mlp-root',str(MLP),'--blend-root',str(BLEND),'--hurdle-replay',str(HURDLE_REPLAY)]
    with Path('reports/mlp_blend_candidate_training.log').open('w') as log:
        process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in process.stdout:
            print(line,end='',flush=True)
            log.write(line)
            log.flush()
        status=process.wait()
    with tarfile.open('/kaggle/working/mlp_blend_candidate_results.tar.gz','w:gz') as archive:
        for folder in ('reports','artifacts/mlp_blend_candidate'):
            if Path(folder).exists():
                archive.add(folder,arcname=folder)
    if status:
        raise RuntimeError('Fixed audit failed; inspect saved logs/checkpoints')
else:
    print('Review mode: no fitting or calibration.')


## Saved metrics and limitations

MAE uses original units, WAPE is a ratio. Report every horizon and full original population. Keep negative results, inspect zero false-positive rates and independently verify checkpoint/forecast consistency after download. Repeated validation cannot prove no overfitting or future performance. Production is unchanged.


In [ ]:
paths=[Path('reports/mlp_blend_candidate_validation.csv'),Path('../reports/mlp_blend_candidate_validation.csv')]
if Path('/kaggle/input').is_dir():
    paths+=list(Path('/kaggle/input').rglob('mlp_blend_candidate_validation.csv'))
path=next((p for p in paths if p.exists()),None)
if path:
    import pandas as pd
    from IPython.display import display
    frame=pd.read_csv(path)
    display(frame.loc[frame.load_tier=='All',['model','horizon','mae','wape','n_valid']])
else:
    print('No completed audit scores attached.')


## Completed v1 audit: lower MAE, accuracy target still unmet

The fixed ten-epoch MLP-B refit and unchanged 20% mixture produce MAE
0.226480 / 0.249664 / 0.248324 at 1/6/24h. WAPE is 40.80% / 44.28% / 44.79%.
Every point MAE is lower than the prior Linear blend and other selected replay
cases on the common validation population. Strict zero false-positive rates
worsen to 99.84% / 99.93% / 99.97%; no risk-quality gain is claimed.

Six artifact files, pinned protocol/provenance, full replay arithmetic and all
reported full-population metrics were verified. Eighty-one history-only
component predictions match cloud output within declared tolerances; maximum
error is 9.20e-8 original units. Original Linear-blend verification is unchanged.

Validation has been reused repeatedly. Spatial intervals do not prove future
performance or no overfitting; the 6h interval versus NR3500 crosses zero.
WAPE <10% remains unmet. Retain the frozen coefficient/epoch rule and do not
promote production or replay the original holdout from this result.

Review `reports/mlp_blend_candidate_analysis.md`, comparison, intervals and
verification JSON. Default Run All performs no refit. The hosted execution
snapshot remains unchanged.
